In [1]:
import pandas as pd
import numpy as np

In [5]:
DATA_PATH = "../data/processed/forecast_dataset.csv"

df = pd.read_csv(DATA_PATH)

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values(
    ["Outlet", "ProductId", "Date"]
).reset_index(drop=True)

print("Shape:", df.shape)
print("Date:", df["Date"].min(), "->", df["Date"].max())
print("Outlets:", df["Outlet"].nunique())
print("Products:", df["ProductId"].nunique())

display(df.head())

Shape: (12240, 9)
Date: 2025-01-02 00:00:00 -> 2025-09-30 00:00:00
Outlets: 3
Products: 15


,Date,Outlet,ProductId,ProductName,Variant,Category,QuantitySold,DayOfWeek,DayOfWeekNum
0,2025-01-02,SHOP001,5,Americano,Ice Houesblend,Coffee,1,Thursday,3
1,2025-01-03,SHOP001,5,Americano,Ice Houesblend,Coffee,1,Friday,4
2,2025-01-04,SHOP001,5,Americano,Ice Houesblend,Coffee,0,Saturday,5
3,2025-01-05,SHOP001,5,Americano,Ice Houesblend,Coffee,1,Sunday,6
4,2025-01-06,SHOP001,5,Americano,Ice Houesblend,Coffee,3,Monday,0


In [6]:
# Check duplicates
duplicates = df.duplicated(
    subset=["Date", "Outlet", "ProductId"]
).sum()

# Check missing target
missing_target = df["QuantitySold"].isna().sum()

print("Duplicates:", duplicates)
print("Missing target:", missing_target)

assert duplicates == 0
assert missing_target == 0
assert df["Outlet"].nunique() == 3
assert df["ProductId"].nunique() == 15

Duplicates: 0
Missing target: 0


In [7]:
# Calendar features
df["day_of_week"] = df["Date"].dt.dayofweek
df["day_of_month"] = df["Date"].dt.day
df["month"] = df["Date"].dt.month
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

In [9]:
# Historical features

group_keys = ["Outlet", "ProductId"]

for lag in [1, 2, 3, 7, 14]:
    df[f"lag_{lag}"] = (
        df
        .groupby(group_keys)["QuantitySold"]
        .shift(lag)
    )

df["rolling_mean_7"] = (
    df.groupby(group_keys)["QuantitySold"]
      .transform(
          lambda x: x.shift(1).rolling(7).mean()
      )
)

df["rolling_mean_14"] = (
    df.groupby(group_keys)["QuantitySold"]
      .transform(
          lambda x: x.shift(1).rolling(14).mean()
      )
)

df["rolling_sum_7"] = (
    df
    .groupby(group_keys)["QuantitySold"]
    .transform(
        lambda x: x.shift(1).rolling(7).sum()
    )
)

In [10]:
# Intermittency features

df["zero_days_7"] = (
    df
    .groupby(group_keys)["QuantitySold"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(7)
        .apply(
            lambda window: (window == 0).sum(),
            raw=True
        )
    )
)

In [11]:
df["days_since_sale"] = (
    df
    .groupby(group_keys)["QuantitySold"]
    .transform(
        lambda x:
        x.shift(1)
        .eq(0)
        .groupby(
            x.shift(1).ne(0).cumsum()
        )
        .cumsum()
    )
)

In [27]:
# Kolom yang akan disimpan ke model_dataset.csv
FINAL_COLUMNS = [
    # Metadata
    "Date",
    "Outlet",
    "ProductId",
    "ProductName",
    "Variant",
    "Category",

    # Target
    "QuantitySold",

    # Calendar Features
    "day_of_week",
    "day_of_month",
    "month",
    "is_weekend",

    # Historical Demand Features
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_7",
    "lag_14",

    # Rolling Demand Features
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_sum_7",

    # Intermittency Feature
    "zero_days_7"
]

# Predictor yang nantinya digunakan model
FEATURE_COLUMNS = [
    "Outlet",
    "ProductId",

    "day_of_week",
    "day_of_month",
    "month",
    "is_weekend",

    "lag_1",
    "lag_2",
    "lag_3",
    "lag_7",
    "lag_14",

    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_sum_7",

    "zero_days_7"
]

TARGET = "QuantitySold"

In [29]:
model_df = df[FINAL_COLUMNS].copy()

print("Shape:", model_df.shape)

Shape: (12240, 20)


In [31]:
# Remove warm-up rows (rows with NaN values in any of the feature columns)

model_df = (
    model_df
    .dropna(subset=FEATURE_COLUMNS)
    .reset_index(drop=True)
)

print("Final shape:", model_df.shape)
print(
    "Date range:",
    model_df["Date"].min(),
    "->",
    model_df["Date"].max()
)

Final shape: (11610, 20)
Date range: 2025-01-16 00:00:00 -> 2025-09-30 00:00:00


In [39]:
#Select only the final columns & remove warm-up rows (rows with NaN values in any of the feature columns) 

model_df = df[FINAL_COLUMNS].copy()

model_df = model_df.dropna(
    subset=FEATURE_COLUMNS
).reset_index(drop=True)

print("Before:", len(df))
print("After :", len(model_df))
print("Removed:", len(df) - len(model_df))
print("Final shape:", model_df.shape)

Before: 12240
After : 11610
Removed: 630
Final shape: (11610, 20)


In [40]:
#Final validation

print("Rows       :", len(model_df))
print("Columns    :", len(model_df.columns))
print("Missing    :", model_df.isna().sum().sum())

duplicates = model_df.duplicated(
    subset=["Date", "Outlet", "ProductId"]
).sum()

print("Duplicates :", duplicates)
print("Outlets    :", model_df["Outlet"].nunique())
print("Products   :", model_df["ProductId"].nunique())

assert model_df.isna().sum().sum() == 0
assert duplicates == 0
assert model_df["Outlet"].nunique() == 3
assert model_df["ProductId"].nunique() == 15

print("\nFINAL VALIDATION: PASSED")

Rows       : 11610
Columns    : 20
Missing    : 0
Duplicates : 0
Outlets    : 3
Products   : 15

FINAL VALIDATION: PASSED


In [41]:
#Feature sanity check

sample = model_df[
    (model_df["Outlet"] == "SHOP001") &
    (model_df["ProductId"] == 5)
][
    [
        "Date",
        "QuantitySold",
        "lag_1",
        "lag_7",
        "rolling_mean_7",
        "zero_days_7"
    ]
].head(10)

display(sample)

,Date,QuantitySold,lag_1,lag_7,rolling_mean_7,zero_days_7
0,2025-01-16,2,2.0,0.0,1.285714,2.0
1,2025-01-17,1,2.0,1.0,1.571429,1.0
2,2025-01-18,1,1.0,1.0,1.571429,1.0
3,2025-01-19,0,1.0,3.0,1.571429,1.0
4,2025-01-20,0,0.0,0.0,1.142857,2.0
5,2025-01-21,3,0.0,2.0,1.142857,2.0
6,2025-01-22,2,3.0,2.0,1.285714,2.0
7,2025-01-23,1,2.0,2.0,1.285714,2.0
8,2025-01-24,0,1.0,1.0,1.142857,2.0
9,2025-01-25,1,0.0,1.0,1.000000,3.0


In [42]:
# Export to CSV

OUTPUT_PATH = "../data/processed/model_dataset.csv"

model_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)
print("Final shape:", model_df.shape)

Saved: ../data/processed/model_dataset.csv
Final shape: (11610, 20)


In [43]:
# Load zero-filled panel
df = pd.read_csv(
    "../data/processed/forecast_dataset.csv"
)

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values(
    ["Outlet", "ProductId", "Date"]
).reset_index(drop=True)

print(df.shape)
display(df.head())

(12240, 9)


,Date,Outlet,ProductId,ProductName,Variant,Category,QuantitySold,DayOfWeek,DayOfWeekNum
0,2025-01-02,SHOP001,5,Americano,Ice Houesblend,Coffee,1,Thursday,3
1,2025-01-03,SHOP001,5,Americano,Ice Houesblend,Coffee,1,Friday,4
2,2025-01-04,SHOP001,5,Americano,Ice Houesblend,Coffee,0,Saturday,5
3,2025-01-05,SHOP001,5,Americano,Ice Houesblend,Coffee,1,Sunday,6
4,2025-01-06,SHOP001,5,Americano,Ice Houesblend,Coffee,3,Monday,0


In [44]:
# ============================================================
# DEMAND PATTERN CLASSIFICATION — ADI & CV²
# ============================================================

demand_patterns = []

for (outlet, product_id), group in df.groupby(
    ["Outlet", "ProductId"]
):

    group = group.sort_values("Date")

    total_periods = len(group)

    # Ambil hanya hari ketika benar-benar ada penjualan
    positive_demand = group.loc[
        group["QuantitySold"] > 0,
        "QuantitySold"
    ]

    nonzero_periods = len(positive_demand)

    # -----------------------------
    # ADI
    # -----------------------------
    if nonzero_periods > 0:
        adi = total_periods / nonzero_periods
    else:
        adi = np.nan

    # -----------------------------
    # CV²
    # -----------------------------
    if nonzero_periods > 1:
        mean_demand = positive_demand.mean()
        std_demand = positive_demand.std(ddof=1)

        cv_squared = (
            std_demand / mean_demand
        ) ** 2
    else:
        cv_squared = np.nan

    # Metadata
    product_name = group["ProductName"].iloc[0]
    variant = group["Variant"].iloc[0]

    demand_patterns.append({
        "Outlet": outlet,
        "ProductId": product_id,
        "ProductName": product_name,
        "Variant": variant,
        "TotalPeriods": total_periods,
        "NonZeroPeriods": nonzero_periods,
        "ADI": adi,
        "CV2": cv_squared
    })

demand_pattern_df = pd.DataFrame(demand_patterns)

display(
    demand_pattern_df
    .sort_values(["Outlet", "ProductId"])
)

,Outlet,ProductId,ProductName,Variant,TotalPeriods,NonZeroPeriods,ADI,CV2
0,SHOP001,5,Americano,Ice Houesblend,272,197,1.380711,0.269919
1,SHOP001,6,Americano,Hot Arabica,272,189,1.439153,0.319458
2,SHOP001,8,Apple Pie Latte,-,272,175,1.554286,0.328131
3,SHOP001,15,Basic Latte,Hot Arabica,272,187,1.454545,0.256648
4,SHOP001,18,Basic Latte,Ice Arabica,272,206,1.320388,0.301039
5,SHOP001,25,Butterscotch,-,272,187,1.454545,0.319750
6,SHOP001,30,Cappuccino,Hot Arabica,272,190,1.431579,0.266316
7,SHOP001,43,Espresso Houseblend,Basic Espresso,272,194,1.402062,0.297202
8,SHOP001,49,Friendly Coffee,Hot,272,187,1.454545,0.245316
9,SHOP001,50,Friendly Coffee,Ice,272,197,1.380711,0.255477


In [45]:
# ============================================================
# CLASSIFY DEMAND PATTERN
# ============================================================

ADI_THRESHOLD = 1.32
CV2_THRESHOLD = 0.49


def classify_demand(row):

    adi = row["ADI"]
    cv2 = row["CV2"]

    if pd.isna(adi) or pd.isna(cv2):
        return "Unknown"

    if adi < ADI_THRESHOLD and cv2 < CV2_THRESHOLD:
        return "Smooth"

    elif adi >= ADI_THRESHOLD and cv2 < CV2_THRESHOLD:
        return "Intermittent"

    elif adi < ADI_THRESHOLD and cv2 >= CV2_THRESHOLD:
        return "Erratic"

    else:
        return "Lumpy"


demand_pattern_df["DemandType"] = (
    demand_pattern_df.apply(
        classify_demand,
        axis=1
    )
)

display(
    demand_pattern_df[
        [
            "Outlet",
            "ProductId",
            "ProductName",
            "Variant",
            "ADI",
            "CV2",
            "DemandType"
        ]
    ].sort_values(["Outlet", "ProductId"])
)

,Outlet,ProductId,ProductName,Variant,ADI,CV2,DemandType
0,SHOP001,5,Americano,Ice Houesblend,1.380711,0.269919,Intermittent
1,SHOP001,6,Americano,Hot Arabica,1.439153,0.319458,Intermittent
2,SHOP001,8,Apple Pie Latte,-,1.554286,0.328131,Intermittent
3,SHOP001,15,Basic Latte,Hot Arabica,1.454545,0.256648,Intermittent
4,SHOP001,18,Basic Latte,Ice Arabica,1.320388,0.301039,Intermittent
5,SHOP001,25,Butterscotch,-,1.454545,0.319750,Intermittent
6,SHOP001,30,Cappuccino,Hot Arabica,1.431579,0.266316,Intermittent
7,SHOP001,43,Espresso Houseblend,Basic Espresso,1.402062,0.297202,Intermittent
8,SHOP001,49,Friendly Coffee,Hot,1.454545,0.245316,Intermittent
9,SHOP001,50,Friendly Coffee,Ice,1.380711,0.255477,Intermittent


In [46]:
pattern_summary = (
    demand_pattern_df["DemandType"]
    .value_counts()
    .rename_axis("DemandType")
    .reset_index(name="Count")
)

pattern_summary["Percentage"] = (
    pattern_summary["Count"]
    / len(demand_pattern_df)
    * 100
).round(2)

display(pattern_summary)

,DemandType,Count,Percentage
0,Intermittent,45,100.0


In [48]:
display(
    demand_pattern_df[
        [
            "Outlet",
            "ProductId",
            "ProductName",
            "Variant",
            "ADI",
            "CV2",
            "DemandType"
        ]
    ].sort_values("ADI")
)

,Outlet,ProductId,ProductName,Variant,ADI,CV2,DemandType
4,SHOP001,18,Basic Latte,Ice Arabica,1.320388,0.301039,Intermittent
0,SHOP001,5,Americano,Ice Houesblend,1.380711,0.269919,Intermittent
9,SHOP001,50,Friendly Coffee,Ice,1.380711,0.255477,Intermittent
13,SHOP001,80,Sitrus Cafe,-,1.394872,0.273287,Intermittent
7,SHOP001,43,Espresso Houseblend,Basic Espresso,1.402062,0.297202,Intermittent
10,SHOP001,56,Happy Moca,Hot,1.409326,0.236121,Intermittent
12,SHOP001,79,Shakencano,-,1.431579,0.306665,Intermittent
6,SHOP001,30,Cappuccino,Hot Arabica,1.431579,0.266316,Intermittent
1,SHOP001,6,Americano,Hot Arabica,1.439153,0.319458,Intermittent
14,SHOP001,88,Vanilla Latte,-,1.454545,0.292217,Intermittent


In [49]:
print("ADI")
print(demand_pattern_df["ADI"].describe())

print("\nCV²")
print(demand_pattern_df["CV2"].describe())

ADI
count    45.000000
mean      1.934870
std       0.459337
min       1.320388
25%       1.454545
50%       1.875862
75%       2.450450
max       2.640777
Name: ADI, dtype: float64

CV²
count    45.000000
mean      0.243204
std       0.048119
min       0.124099
25%       0.210533
50%       0.252765
75%       0.274147
max       0.328131
Name: CV2, dtype: float64
